# LeetCode 3704 — Count No-Zero Pairs That Sum to N bằng Python

- Link: https://leetcode.com/problems/count-no-zero-pairs-that-sum-to-n/
- Độ khó: **Hard**
- Chủ đề: **Digit DP trên phép cộng** (từ hàng đơn vị, có **nhớ**), trạng thái "số còn chữ số không", đếm theo độ dài

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countNoZeroPairs(self, n: int) -> int:
```

## 1. Mục tiêu học

1. Mô phỏng phép cộng `a + b = n` **từng chữ số từ phải sang trái** với biến **nhớ** `carry ∈ {0, 1}`.
2. Hiểu ràng buộc "không có chữ số 0": khi một số **còn** chữ số, chữ số đó thuộc `1..9`; khi số đã **hết** (vượt quá độ dài), các vị trí sau là 0 "ảo" và không được "sống lại".
3. Thiết kế trạng thái `(carry, aliveA, aliveB)` — 8 trạng thái mỗi vị trí.
4. So sánh với cách **cố định độ dài** `(la, lb)` rồi DP chỉ theo `carry`.
5. Kiểm chứng bằng vét cạn cho `n` nhỏ.

## 2. Đề bài & ví dụ

Số **không-chứa-0** là số nguyên dương không có chữ số 0. Đếm số cặp **có thứ tự** `(a, b)` gồm hai số không-chứa-0 với `a + b = n`.

**Ví dụ 1:** `n = 2` → `1` (`(1,1)`).

**Ví dụ 2:** `n = 3` → `2`.

**Ví dụ 3:** `n = 11` → `8` (`(1,10)` bị loại vì `10` chứa 0).

**Ràng buộc:** `2 ≤ n ≤ 10^15`.

In [1]:
from functools import lru_cache
import random, time

examples = [(2, 1), (3, 2), (11, 8)]
for n, exp in examples:
    print(n, "-> kỳ vọng", exp)

2 -> kỳ vọng 1
3 -> kỳ vọng 2
11 -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Phép cộng theo cột

Tại vị trí `i` (tính từ hàng đơn vị): `da + db + carry_in = n_i + 10·carry_out`. Biết `n_i` và `carry_in`, mỗi lựa chọn `(da, db)` xác định duy nhất `carry_out`. Sau chữ số cao nhất của `n`, cần `carry = 0`.

In [2]:
a, b = 58, 67
print(f"{a} + {b} = {a + b}")
carry = 0
for i, (x, y) in enumerate(zip(str(a)[::-1], str(b)[::-1])):
    s = int(x) + int(y) + carry
    print(f"  cột {i}: {x} + {y} + nhớ {carry} = {s} -> viết {s % 10}, nhớ {s // 10}")
    carry = s // 10
print("  nhớ cuối:", carry)

58 + 67 = 125
  cột 0: 8 + 7 + nhớ 0 = 15 -> viết 5, nhớ 1
  cột 1: 5 + 6 + nhớ 1 = 12 -> viết 2, nhớ 1
  nhớ cuối: 1


### 3.2. "Sống" và "hết"

Số `a` có `la` chữ số: các cột `0..la−1` là chữ số thật (`1..9`), các cột `≥ la` là 0 ảo. Khi DP qua các cột, `aliveA = 1` nghĩa là `a` vẫn còn chữ số ở cột hiện tại. Từ `alive` có thể chuyển sang `hết` (đặt 0 ảo, bắt đầu từ cột 1 trở đi để `a ≥ 1`), nhưng **không** chuyển ngược lại.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Duyệt cột `i = 0, 1, …, L` (thêm một cột 0 phía trên `n` để "hút" phần nhớ cuối). Trạng thái `(carry, aliveA, aliveB)`, số cách tương ứng. Ở mỗi cột:

- `a` sống: chọn `da ∈ 1..9` (vẫn sống) hoặc — nếu `i ≥ 1` — `da = 0` và chuyển sang hết;
- `a` đã hết: `da = 0`;
- tương tự cho `b`;
- giữ lại lựa chọn mà `(da + db + carry) % 10 = n_i`.

Kết thúc: trạng thái `(carry = 0, hết, hết)`.

Ví dụ `n = 11`: cột 0 cần `da + db ≡ 1 (mod 10)` với `da, db ∈ 1..9` ⇒ `da + db = 11` (nhớ 1), 8 cặp; cột 1 cần `da + db + 1 ≡ 1` ⇒ cả hai hết (`0 + 0 + 1 = 1`) ⇒ `8`. ✔

## 5. Cách 1 — Brute Force

In [3]:
def brute(n):
    return sum(1 for a in range(1, n) if "0" not in str(a) and "0" not in str(n - a))

for n, exp in examples:
    print(brute(n), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
8 kỳ vọng 8


**Nút thắt:** `O(n · log n)` — với `10^15` là bất khả thi.

## 6. Ý tưởng thuật toán / bước đột phá

Số cột ≤ 17, mỗi cột 8 trạng thái × tối đa `10 × 10` lựa chọn ⇒ khoảng `17 · 8 · 100 ≈ 14.000` phép — tức thời.

Điểm tinh tế:

- **Cột 0** không được "hết" (cả `a`, `b` đều ≥ 1 ⇒ có ít nhất một chữ số).
- Cần một cột phụ phía trên để phần nhớ cuối được "trả" đúng: ở cột phụ `n_L = 0`, cả hai phải đã hết (vì nếu còn sống chữ số `≥ 1` thì tổng vượt `n`), và `carry` vào phải là 0.

## 7. Các cách tiếp theo

- **Cách 2:** cố định độ dài `(la, lb)`, DP chỉ theo `carry` — `O(L² · L · 100)`.
- **Cách 3:** DP lặp với trạng thái `(carry, aliveA, aliveB)` (lời giải nộp).
- **Cách 4:** đệ quy có memo từ hàng đơn vị — cùng trạng thái, viết dạng hàm.

### 7.1. Cách 2 — Cố định độ dài

In [4]:
def by_lengths(n):
    digits = list(map(int, str(n)))[::-1]; L = len(digits)
    total = 0
    for la in range(1, L + 1):
        for lb in range(1, L + 1):
            ways = {0: 1}                                 # carry -> số cách
            for i in range(L):
                A = range(1, 10) if i < la else (0,)
                B = range(1, 10) if i < lb else (0,)
                nw = {}
                for c, w in ways.items():
                    for da in A:
                        for db in B:
                            s = da + db + c
                            if s % 10 == digits[i]:
                                nw[s // 10] = nw.get(s // 10, 0) + w
                ways = nw
            total += ways.get(0, 0)                       # không được còn nhớ sau chữ số cao nhất
    return total

for n, exp in examples:
    print(by_lengths(n), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
8 kỳ vọng 8


### 7.2. Cách 3 — DP lặp `(carry, aliveA, aliveB)` (lời giải chính)

In [5]:
def dp_alive(n):
    digits = list(map(int, str(n)))[::-1] + [0]      # thêm cột phụ
    dp = {(0, 1, 1): 1}
    for i, t in enumerate(digits):
        nd = {}
        for (c, sa, sb), w in dp.items():
            A = [(d, 1) for d in range(1, 10)] if sa else []
            if not sa or i > 0: A.append((0, 0))      # hết (hoặc đã hết)
            B = [(d, 1) for d in range(1, 10)] if sb else []
            if not sb or i > 0: B.append((0, 0))
            for da, na in A:
                for db, nb in B:
                    s = da + db + c
                    if s % 10 == t:
                        key = (s // 10, na, nb); nd[key] = nd.get(key, 0) + w
        dp = nd
    return dp.get((0, 0, 0), 0)

for n, exp in examples:
    print(dp_alive(n), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
8 kỳ vọng 8


### 7.3. Cách 4 — Đệ quy có memo

In [6]:
def dp_recursive(n):
    digits = list(map(int, str(n)))[::-1] + [0]

    @lru_cache(None)
    def go(i, c, sa, sb):
        if i == len(digits):
            return 1 if (c == 0 and not sa and not sb) else 0
        res = 0
        for da, na in ([(d, 1) for d in range(1, 10)] if sa else []) + ([(0, 0)] if (not sa or i > 0) else []):
            for db, nb in ([(d, 1) for d in range(1, 10)] if sb else []) + ([(0, 0)] if (not sb or i > 0) else []):
                s = da + db + c
                if s % 10 == digits[i]:
                    res += go(i + 1, s // 10, na, nb)
        return res

    return go(0, 0, 1, 1)

for n, exp in examples:
    print(dp_recursive(n), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
8 kỳ vọng 8


## 8. TRACE từng bước

In [7]:
def trace(n):
    digits = list(map(int, str(n)))[::-1] + [0]
    dp = {(0, 1, 1): 1}
    print("chữ số của n (từ hàng đơn vị) + cột phụ:", digits)
    for i, t in enumerate(digits):
        nd = {}
        for (c, sa, sb), w in dp.items():
            A = [(d, 1) for d in range(1, 10)] if sa else []
            if not sa or i > 0: A.append((0, 0))
            B = [(d, 1) for d in range(1, 10)] if sb else []
            if not sb or i > 0: B.append((0, 0))
            for da, na in A:
                for db, nb in B:
                    s = da + db + c
                    if s % 10 == t:
                        key = (s // 10, na, nb); nd[key] = nd.get(key, 0) + w
        dp = nd
        print(f"  sau cột {i} (n_i = {t}): " + ", ".join(f"(nhớ={c}, a{'sống' if a else 'hết'}, b{'sống' if b else 'hết'}): {w}" for (c, a, b), w in sorted(dp.items())))
    print("đáp án:", dp.get((0, 0, 0), 0), "| brute:", brute(n))

trace(11)

chữ số của n (từ hàng đơn vị) + cột phụ: [1, 1, 0]
  sau cột 0 (n_i = 1): (nhớ=1, asống, bsống): 8
  sau cột 1 (n_i = 1): (nhớ=0, ahết, bhết): 8, (nhớ=1, asống, bsống): 72
  sau cột 2 (n_i = 0): (nhớ=0, ahết, bhết): 8, (nhớ=1, ahết, bsống): 72, (nhớ=1, asống, bhết): 72, (nhớ=1, asống, bsống): 576
đáp án: 8 | brute: 8


In [8]:
trace(105)

chữ số của n (từ hàng đơn vị) + cột phụ: [5, 0, 1, 0]
  sau cột 0 (n_i = 5): (nhớ=0, asống, bsống): 4, (nhớ=1, asống, bsống): 4
  sau cột 1 (n_i = 0): (nhớ=0, ahết, bhết): 4, (nhớ=1, ahết, bsống): 4, (nhớ=1, asống, bhết): 4, (nhớ=1, asống, bsống): 68
  sau cột 2 (n_i = 1): (nhớ=0, ahết, bhết): 76, (nhớ=1, asống, bsống): 612
  sau cột 3 (n_i = 0): (nhớ=0, ahết, bhết): 76, (nhớ=1, ahết, bsống): 612, (nhớ=1, asống, bhết): 612, (nhớ=1, asống, bsống): 4896
đáp án: 76 | brute: 76


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def countNoZeroPairs(self, n: int) -> int:
        digits = list(map(int, str(n)))[::-1] + [0]
        dp = [[[0, 0], [0, 0]], [[0, 0], [0, 0]]]           # dp[carry][aliveA][aliveB]
        dp[0][1][1] = 1
        for i, t in enumerate(digits):
            nd = [[[0, 0], [0, 0]], [[0, 0], [0, 0]]]
            for c in range(2):
                for sa in range(2):
                    for sb in range(2):
                        w = dp[c][sa][sb]
                        if not w:
                            continue
                        A = [(d, 1) for d in range(1, 10)] if sa else []
                        if not sa or i > 0:
                            A.append((0, 0))
                        B = [(d, 1) for d in range(1, 10)] if sb else []
                        if not sb or i > 0:
                            B.append((0, 0))
                        for da, na in A:
                            for db, nb in B:
                                s = da + db + c
                                if s % 10 == t:
                                    nd[s // 10][na][nb] += w
            dp = nd
        return dp[0][0][0]

sol = Solution()
for n, exp in examples:
    got = sol.countNoZeroPairs(n)
    print(n, "->", got, "✔" if got == exp else "✘")

2 -> 1 ✔
3 -> 2 ✔
11 -> 8 ✔


## 10. Vì sao thuật toán đúng

1. **Song ánh:** mỗi cặp `(a, b)` có độ dài `la, lb ≤ L` ứng với đúng một dãy lựa chọn theo cột: ở cột `i < la`, `da = a_i ∈ 1..9` và `a` "sống"; ở cột `i ≥ la`, `da = 0` và `a` "hết" (chuyển trạng thái đúng một lần tại cột `la ≥ 1`). Tương tự `b`.
2. **Điều kiện tổng:** `a + b = n` ⇔ mọi cột khớp `(da + db + carry_in) % 10 = n_i` với `carry` lan truyền đúng **và** không còn nhớ sau cột cao nhất. Cột phụ (`n_L = 0`) làm đúng việc đó: nếu `carry_in = 1` thì cần `da + db + 1 ≡ 0`, buộc ít nhất một số còn sống ở cột này hoặc sinh nhớ mới — cả hai đều bị loại vì ta chỉ nhận trạng thái cuối `(carry = 0, hết, hết)`; còn số dài đúng `L` chữ số được phép chuyển sang "hết" tại cột phụ.
3. **Không chứa 0:** chữ số thật chỉ chọn trong `1..9`; 0 chỉ xuất hiện như 0 ảo sau khi hết.
4. **`a, b ≥ 1`:** cột 0 cấm "hết" ⇒ mỗi số có ít nhất một chữ số thật.

## 11. Phân tích Time / Space Complexity

Với `L ≤ 16` chữ số:

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n log n)` | `O(1)` |
| 2. Cố định độ dài | `O(L³ · 100)` | `O(1)` |
| 3. DP lặp | `O(L · 8 · 100)` | `O(1)` |
| 4. Đệ quy memo | `O(L · 8 · 100)` | `O(L · 8)` |

## 12. Bộ test

In [10]:
METHODS = [brute, by_lengths, dp_alive, dp_recursive, lambda n: Solution().countNoZeroPairs(n)]
for n, exp in examples:
    assert all(f(n) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 toàn bộ n = 2..5000 so với vét cạn
for n in range(2, 5001):
    b = brute(n)
    assert Solution().countNoZeroPairs(n) == b == dp_alive(n), n
for n in [10, 100, 1000, 101, 110, 1010]:
    assert by_lengths(n) == dp_recursive(n) == brute(n)
print("n = 2..5000: OK")

n = 2..5000: OK


In [12]:
# 12.3 n lớn ngẫu nhiên: các cách nhanh phải khớp
rng = random.Random(3704)
for _ in range(300):
    n = rng.randint(2, 10**15)
    r = [f(n) for f in METHODS[1:]]
    assert len(set(r)) == 1, (n, r)
print("300 n lớn: OK")

300 n lớn: OK


In [13]:
# 12.4 thời gian
t0 = time.time()
for _ in range(1000): Solution().countNoZeroPairs(10**15)
print("1000 lần với n = 10^15:", round(time.time() - t0, 2), "s | đáp án:", Solution().countNoZeroPairs(10**15),
      "| n = 999999999999999:", Solution().countNoZeroPairs(999999999999999))

1000 lần với n = 10^15: 0.18 s | đáp án: 50891681057058 | n = 999999999999999: 45237049828496


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép "hết" ngay ở cột 0 (đếm cả `a = 0`)

In [14]:
def wrong_allow_zero_len(n):
    digits = list(map(int, str(n)))[::-1] + [0]; dp = {(0, 1, 1): 1}
    for i, t in enumerate(digits):
        nd = {}
        for (c, sa, sb), w in dp.items():
            A = ([(d, 1) for d in range(1, 10)] if sa else []) + [(0, 0)]          # SAI: kể cả i = 0
            B = ([(d, 1) for d in range(1, 10)] if sb else []) + [(0, 0)]
            for da, na in A:
                for db, nb in B:
                    s = da + db + c
                    if s % 10 == t: nd[(s // 10, na, nb)] = nd.get((s // 10, na, nb), 0) + w
        dp = nd
    return dp.get((0, 0, 0), 0)

print("sai:", wrong_allow_zero_len(9), "| đúng:", brute(9))

sai: 10 | đúng: 8


### Lỗi 2 — Cho số đã hết "sống lại"

Nếu ở trạng thái hết vẫn cho chọn chữ số `1..9`, ta đếm cả những số có 0 ở giữa (ví dụ `105` được ghép từ "1", "0 ảo", "5").

In [15]:
def wrong_revive(n):
    digits = list(map(int, str(n)))[::-1] + [0]; dp = {(0, 1, 1): 1}
    for i, t in enumerate(digits):
        nd = {}
        for (c, sa, sb), w in dp.items():
            A = [(d, 1) for d in range(1, 10)] + ([(0, 0)] if i > 0 else [])           # SAI: bỏ qua sa
            B = [(d, 1) for d in range(1, 10)] + ([(0, 0)] if i > 0 else [])
            for da, na in A:
                for db, nb in B:
                    s = da + db + c
                    if s % 10 == t: nd[(s // 10, na, nb)] = nd.get((s // 10, na, nb), 0) + w
        dp = nd
    return dp.get((0, 0, 0), 0)

print("sai:", wrong_revive(1000), "| đúng:", brute(1000))

sai: 900 | đúng: 738


### Lỗi 3 — Bỏ cột phụ nhưng vẫn đòi trạng thái "cả hai đã hết"

Không có cột phụ thì số có **cùng độ dài với `n`** (ví dụ `(11, 11)` với `n = 22`) vẫn đang "sống" sau cột cuối và bị bỏ sót. Cột phụ cho phép chúng chuyển sang "hết" một cách hợp lệ.

In [16]:
def wrong_no_extra(n):
    digits = list(map(int, str(n)))[::-1]; dp = {(0, 1, 1): 1}             # SAI: thiếu cột phụ
    for i, t in enumerate(digits):
        nd = {}
        for (c, sa, sb), w in dp.items():
            A = [(d, 1) for d in range(1, 10)] if sa else []
            if not sa or i > 0: A.append((0, 0))
            B = [(d, 1) for d in range(1, 10)] if sb else []
            if not sb or i > 0: B.append((0, 0))
            for da, na in A:
                for db, nb in B:
                    s = da + db + c
                    if s % 10 == t: nd[(s // 10, na, nb)] = nd.get((s // 10, na, nb), 0) + w
        dp = nd
    return dp.get((0, 0, 0), 0)                                               # chỉ nhận cả hai đã "hết"

print("sai:", wrong_no_extra(22), "| đúng:", brute(22))
assert wrong_no_extra(22) != brute(22)

sai: 0 | đúng: 17


### Lỗi 4 — Đếm cặp không có thứ tự

Đề đếm `(a, b)` và `(b, a)` là hai cặp khác nhau (ví dụ 2 có đáp án 2). Chia đôi là sai (và còn lẻ khi `a = b`).

In [17]:
print("chia đôi (sai):", Solution().countNoZeroPairs(3) // 2, "| đúng:", Solution().countNoZeroPairs(3))

chia đôi (sai): 1 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai số có tổng `n` với ràng buộc trên chữ số" ⇒ DP theo cột **từ hàng đơn vị** với **nhớ**.
- Số có độ dài thay đổi ⇒ cờ "sống/hết" (hoặc liệt kê độ dài).
- Luôn thêm cột phụ hoặc kiểm tra `carry = 0` ở cuối.
- Kiểm chứng `n ≤ 5000` bằng vét cạn — rất rẻ.

## 15. Biến thể của bài

1. **LC 1317** (*Convert Integer to the Sum of Two No-Zero Integers*): chỉ cần **một** cặp — duyệt `a` từ 1.
2. **Hiệu** `a − b = n` với ràng buộc chữ số: DP với **mượn** thay cho nhớ.
3. **Cấm thêm một chữ số `k`** (ngoài 0): bỏ `k` khỏi tập `1..9`. (Nếu chỉ cấm `k ≠ 0` mà **cho phép** 0 ở giữa số thì cột 0 thật và 0 ảo lẫn nhau — cần thêm trạng thái "chữ số cao nhất đã đặt khác 0", khó hơn.)
4. **Ba số** `a + b + c = n`: nhớ trong `{0, 1, 2}`.

In [18]:
# Biến thể 3 + 4: k số không chứa chữ số 0 và (tuỳ chọn) không chứa thêm chữ số `bad`, tổng bằng n
from itertools import product

def count_k_numbers(n, k=2, bad=None):
    digits = list(map(int, str(n)))[::-1] + [0, 0]
    allowed = [d for d in range(1, 10) if d != bad]
    dp = {(0, (1,) * k): 1}
    for i, t in enumerate(digits):
        nd = {}
        for (c, alive), w in dp.items():
            opts = []
            for s_ in alive:
                o = [(d, 1) for d in allowed] if s_ else []
                if not s_ or i > 0: o.append((0, 0))
                opts.append(o)
            for combo in product(*opts):
                s = sum(d for d, _ in combo) + c
                if s % 10 == t:
                    key = (s // 10, tuple(a for _, a in combo)); nd[key] = nd.get(key, 0) + w
        dp = nd
    return dp.get((0, (0,) * k), 0)

def brute_k3(n):
    ok = lambda x: "0" not in str(x)
    return sum(1 for a in range(1, n) for b in range(1, n - a) if ok(a) and ok(b) and ok(n - a - b))

assert all(count_k_numbers(n) == brute(n) for n in range(2, 300))
assert all(count_k_numbers(n, 3) == brute_k3(n) for n in range(3, 120))
ok5 = lambda x: "0" not in str(x) and "5" not in str(x)
assert all(count_k_numbers(n, 2, 5) == sum(1 for a in range(1, n) if ok5(a) and ok5(n - a)) for n in range(2, 300))
print("Biến thể 3, 4 OK; ba số tổng 100:", count_k_numbers(100, 3))

Biến thể 3, 4 OK; ba số tổng 100: 3600


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Duyệt `a` | `O(n log n)` | Kiểm chứng |
| 2. Cố định độ dài | DP theo nhớ cho mỗi `(la, lb)` | `O(L³·100)` | Dễ suy luận |
| 3. DP `(carry, alive)` | Gộp độ dài vào trạng thái | `O(L·800)` | **Lời giải chuẩn** |
| 4. Đệ quy memo | Như 3 dạng hàm | `O(L·800)` | Viết nhanh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countNoZeroPairs(10)`.

**Bài 2 (tính tay).** `countNoZeroPairs(20)`.

**Bài 3 (điền TODO).** Viết `find_one_pair(n)` trả về một cặp hợp lệ bất kỳ (LC 1317).

**Bài 4 (tự viết lại).** Đếm số cặp `(a, b)` không-chứa-0 với `a + b = n` **và** `a ≤ b`.

**Bài 5 (mini challenge).** Đếm số cặp `(a, b)` không-chứa-0 với `a − b = n` và `a ≤ 10^15` (gợi ý: DP với "mượn").

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def find_one_pair(n):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_diff_pairs(n, limit_digits=15):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `a + b = 10`, cả hai một chữ số. (2) `a ∈ 1..9`. (3) `9`.

**Bài 2.** (1) `a + b = 20`; hai số một chữ số có tổng tối đa 18 ⇒ một số phải có hai chữ số. (2) `a ∈ 11..19` (không chứa 0) cho `b = 20 − a ∈ 1..9`. (3) 9 cặp, cộng 9 cặp đối xứng ⇒ `18`.

**Bài 3.** (1) Duyệt `a = 1, 2, …`. (2) Dừng khi cả `a` và `n − a` không chứa 0. (3) Luôn tồn tại với `n ≥ 2`.

**Bài 4.** (1) Đếm có thứ tự `T`. (2) Cặp `a = b` có khi `n` chẵn và `n/2` không chứa 0. (3) `(T + [a=b]) / 2`.

**Bài 5.** (1) `a = b + n`: cộng `b + n` theo cột. (2) Trạng thái `(carry, aliveA, aliveB)` nhưng `a` là kết quả: `a_i = (b_i + n_i + c) % 10` phải ≠ 0 khi `a` sống. (3) Giới hạn số cột.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().countNoZeroPairs(10) == 9 == brute(10)
assert Solution().countNoZeroPairs(20) == 18 == brute(20)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def find_one_pair_sol(n):
    a = 1
    while "0" in str(a) or "0" in str(n - a): a += 1
    return [a, n - a]

def count_unordered_sol(n):
    T = Solution().countNoZeroPairs(n)
    same = 1 if n % 2 == 0 and "0" not in str(n // 2) else 0
    return (T + same) // 2

for n in range(2, 2000):
    a, b = find_one_pair_sol(n); assert a + b == n and "0" not in str(a) + str(b)
    assert count_unordered_sol(n) == sum(1 for x in range(1, n // 2 + 1) if "0" not in str(x) and "0" not in str(n - x))
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: a - b = n, cả hai không chứa 0, a có tối đa D chữ số
def count_diff_pairs_sol(n, D=15):
    nd_ = list(map(int, str(n)))[::-1]; nd_ += [0] * (D + 1 - len(nd_))
    dp = {(0, 1, 1): 1}                       # (carry của phép b + n, aSống, bSống)
    for i in range(D + 1):
        t = nd_[i]; ndp = {}
        for (c, sa, sb), w in dp.items():
            B = [(d, 1) for d in range(1, 10)] if sb else []
            if not sb or i > 0: B.append((0, 0))
            for db, nb in B:
                s = db + t + c; da = s % 10
                for na in ([1] if (sa and da != 0) else []) + ([0] if da == 0 and (not sa or i > 0) else []):
                    key = (s // 10, na, nb); ndp[key] = ndp.get(key, 0) + w
        dp = ndp
    return dp.get((0, 0, 0), 0)

def brute_diff(n, D):
    return sum(1 for b in range(1, 10**D) if b + n < 10**D and "0" not in str(b) and "0" not in str(b + n))

for n in range(1, 60):
    assert count_diff_pairs_sol(n, 3) == brute_diff(n, 3), n
print("Bài 5 OK; ví dụ n = 5, a ≤ 3 chữ số:", count_diff_pairs_sol(5, 3))

Bài 5 OK; ví dụ n = 5, a ≤ 3 chữ số: 688


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
digits = chữ số của n từ hàng đơn vị + [0]      # cột phụ
dp[(carry=0, aliveA=1, aliveB=1)] = 1
cột i: sống -> chọn 1..9 (vẫn sống) hoặc 0 (hết, chỉ khi i > 0); đã hết -> chỉ 0
       giữ lựa chọn (da + db + carry) % 10 == n_i, carry mới = //10
return dp[(0, 0, 0)]
```

**Checklist:**

- [ ] Cột 0 không cho "hết"?
- [ ] Số đã hết không sống lại?
- [ ] Có cột phụ / kiểm tra nhớ cuối = 0?
- [ ] Đếm cặp **có thứ tự**?
- [ ] Đã so với vét cạn `n ≤ 5000`?